In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
if torch.cuda.is_available():
    # Print the name of the GPU
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
    # Print the number of available CUDA devices
    print(f"Number of GPUs available: {torch.cuda.device_count()}")
else:
    print("No GPU available. Training will run on CPU.")


No GPU available. Training will run on CPU.


In [3]:
batch_size = 2
num_tokens = 3
d_model = 8
num_heads = 2
head_dim = 4

In [4]:
w_q = nn.Linear(d_model, d_model)
w_k = nn.Linear(d_model, d_model)
w_v = nn.Linear(d_model, d_model)
w_o = nn.Linear(d_model, d_model)

In [6]:
dummy_input = torch.rand(batch_size, num_tokens, d_model)
print(dummy_input.shape)
print("(batch_size, num_tokens, embedding_dimension)")
dummy_input

torch.Size([2, 3, 8])
(batch_size, num_tokens, embedding_dimension)


tensor([[[0.6473, 0.5123, 0.7628, 0.1121, 0.8092, 0.7845, 0.3298, 0.0644],
         [0.7531, 0.1591, 0.1561, 0.3592, 0.1829, 0.2017, 0.2987, 0.0312],
         [0.6156, 0.6829, 0.8441, 0.2239, 0.8016, 0.8658, 0.6101, 0.8950]],

        [[0.5825, 0.2182, 0.1473, 0.2879, 0.3891, 0.4254, 0.7122, 0.5119],
         [0.8856, 0.1034, 0.0569, 0.0296, 0.6677, 0.0417, 0.5591, 0.0581],
         [0.7428, 0.9408, 0.2456, 0.3634, 0.0178, 0.4265, 0.4578, 0.2138]]])

In [7]:
q = w_q(dummy_input)
k = w_k(dummy_input)
v = w_v(dummy_input)

In [8]:
print(q.shape)
print("(batch_size, num_tokens, key_dimension)")
q

torch.Size([2, 3, 8])
(batch_size, num_tokens, key_dimension)


tensor([[[-0.2677,  0.1101, -0.2011,  0.0141,  0.7407, -0.5456,  0.5580,
           0.3278],
         [-0.1879, -0.0226, -0.4379, -0.2071,  0.3825, -0.3829,  0.4573,
           0.2011],
         [-0.4323,  0.1380, -0.0655, -0.1773,  0.9670, -0.5746,  0.8292,
           0.4336]],

        [[-0.2607, -0.0342, -0.3636, -0.3236,  0.5631, -0.4652,  0.5859,
           0.1846],
         [-0.1649,  0.0067, -0.2930, -0.3809,  0.3694, -0.5879,  0.3440,
           0.0804],
         [-0.3806,  0.1301, -0.4258,  0.0191,  0.6640, -0.5600,  0.7654,
           0.3137]]], grad_fn=<ViewBackward0>)

In [9]:
q_head = q.view(batch_size, num_tokens, num_heads, head_dim)
k_head = k.view(batch_size, num_tokens, num_heads, head_dim)
v_head = v.view(batch_size, num_tokens, num_heads, head_dim)
print(q_head.shape)
q_head

torch.Size([2, 3, 2, 4])


tensor([[[[-0.2677,  0.1101, -0.2011,  0.0141],
          [ 0.7407, -0.5456,  0.5580,  0.3278]],

         [[-0.1879, -0.0226, -0.4379, -0.2071],
          [ 0.3825, -0.3829,  0.4573,  0.2011]],

         [[-0.4323,  0.1380, -0.0655, -0.1773],
          [ 0.9670, -0.5746,  0.8292,  0.4336]]],


        [[[-0.2607, -0.0342, -0.3636, -0.3236],
          [ 0.5631, -0.4652,  0.5859,  0.1846]],

         [[-0.1649,  0.0067, -0.2930, -0.3809],
          [ 0.3694, -0.5879,  0.3440,  0.0804]],

         [[-0.3806,  0.1301, -0.4258,  0.0191],
          [ 0.6640, -0.5600,  0.7654,  0.3137]]]], grad_fn=<ViewBackward0>)

In [10]:
q_head = q_head.transpose(1, 2)
k_head = k_head.transpose(1, 2)
v_head = v_head.transpose(1, 2)
print(q_head.shape)
print("(batch_size, num_heads, num_tokens, head_dim)")
q_head

torch.Size([2, 2, 3, 4])
(batch_size, num_heads, num_tokens, head_dim)


tensor([[[[-0.2677,  0.1101, -0.2011,  0.0141],
          [-0.1879, -0.0226, -0.4379, -0.2071],
          [-0.4323,  0.1380, -0.0655, -0.1773]],

         [[ 0.7407, -0.5456,  0.5580,  0.3278],
          [ 0.3825, -0.3829,  0.4573,  0.2011],
          [ 0.9670, -0.5746,  0.8292,  0.4336]]],


        [[[-0.2607, -0.0342, -0.3636, -0.3236],
          [-0.1649,  0.0067, -0.2930, -0.3809],
          [-0.3806,  0.1301, -0.4258,  0.0191]],

         [[ 0.5631, -0.4652,  0.5859,  0.1846],
          [ 0.3694, -0.5879,  0.3440,  0.0804],
          [ 0.6640, -0.5600,  0.7654,  0.3137]]]],
       grad_fn=<TransposeBackward0>)

In [11]:
k_t = k_head.transpose(-2, -1)
print(k_t.shape)
k_t

torch.Size([2, 2, 4, 3])


tensor([[[[ 0.2865,  0.1583,  0.4275],
          [-0.1490,  0.1471, -0.4195],
          [-0.0831, -0.1415,  0.1037],
          [ 0.3293,  0.3886,  0.4417]],

         [[ 0.6733,  0.3291,  0.9141],
          [-0.2149, -0.0511, -0.3146],
          [-0.3630, -0.2667, -0.3999],
          [-0.2248,  0.1057, -0.4775]]],


        [[[ 0.2095,  0.2128,  0.4797],
          [-0.0918,  0.0544,  0.0492],
          [ 0.1244, -0.0197, -0.3440],
          [ 0.4020,  0.2767,  0.3806]],

         [[ 0.4522,  0.2909,  0.6362],
          [-0.1539,  0.1011, -0.3555],
          [-0.3839, -0.4659, -0.3893],
          [-0.0090,  0.3012, -0.1666]]]], grad_fn=<TransposeBackward0>)

In [ ]:
sim1 = (q_head @ k_t)/math.sqrt(head_dim)
print(sim1.shape)
print("(batch, num_heads, num_queries, num_keys)")
sim1
# Attention Scores Matrix

torch.Size([2, 2, 3, 3])
(batch, num_heads, num_queries, num_keys


tensor([[[[ 0.0219,  0.1277,  0.0448],
          [ 0.0053,  0.1463,  0.2477],
          [ 0.0056,  0.0811,  0.1028]],

         [[ 0.1455,  0.1557,  0.1516],
          [ 0.0073,  0.0619, -0.0236],
          [ 0.0301,  0.0570, -0.0481]]],


        [[[ 0.0843,  0.0697,  0.1176],
          [ 0.0716,  0.0683,  0.1264],
          [ 0.0647,  0.0276,  0.0658]],

         [[ 0.0849,  0.0578,  0.1704],
          [ 0.0741,  0.0455,  0.1505],
          [ 0.0782,  0.0524,  0.1781]]]], grad_fn=<DivBackward0>)

In [12]:
sim2 = F.softmax(sim1, dim=-1)
print(sim2.shape)
print("(num_batches, num_heads, num_queries, num_keys)")
sim2
# Attention Weights matrix
# Basically building a neural network for value vectors

NameError: name 'sim1' is not defined

In [ ]:
print(v_head.shape)
v_head # Value vectors

torch.Size([2, 2, 3, 4])


tensor([[[[-0.2584,  0.2359, -0.2184, -0.5517],
          [ 0.3617,  0.1351,  0.1894, -0.2387],
          [ 0.0353,  0.0697, -0.0165, -0.1913]],

         [[-0.4173, -0.1015, -0.0277, -0.0166],
          [-0.2465,  0.1930,  0.1727, -0.0116],
          [-0.1047, -0.0286,  0.0677,  0.7633]]],


        [[[ 0.1114,  0.1604,  0.1787, -0.3319],
          [-0.0178,  0.0490,  0.0953, -0.1627],
          [-0.0042, -0.0538,  0.2370, -0.3819]],

         [[-0.2513,  0.1566,  0.1334,  0.1944],
          [ 0.0127, -0.0377,  0.2364,  0.3360],
          [ 0.0695,  0.0403, -0.1851,  0.1830]]]],
       grad_fn=<TransposeBackward0>)

: 

In [ ]:
sim3 = sim2 @ v_head
print(sim3.shape)
sim3 
# Applying Attention Weights to value vectors
# Essentially, passing value vectors through a neural network

torch.Size([2, 2, 3, 4])


tensor([[[[ 0.0574,  0.1459, -0.0079, -0.3231],
          [ 0.0593,  0.1402, -0.0062, -0.3125],
          [ 0.0536,  0.1441, -0.0102, -0.3207]],

         [[-0.2558,  0.0215,  0.0713,  0.2452],
          [-0.2575,  0.0247,  0.0728,  0.2351],
          [-0.2599,  0.0238,  0.0719,  0.2295]]],


        [[[ 0.0297,  0.0507,  0.1714, -0.2938],
          [ 0.0292,  0.0499,  0.1716, -0.2940],
          [ 0.0304,  0.0519,  0.1713, -0.2938]],

         [[-0.0533,  0.0535,  0.0528,  0.2354],
          [-0.0537,  0.0536,  0.0535,  0.2355],
          [-0.0526,  0.0534,  0.0517,  0.2351]]]],
       grad_fn=<UnsafeViewBackward0>)

: 

In [ ]:
sim3 = sim3.transpose(1, 2).contiguous()
sim3 = sim3.view(batch_size, num_tokens, d_model)
print(sim3.shape)
sim3

torch.Size([2, 3, 8])


tensor([[[ 0.0574,  0.0536, -0.2575,  0.1459,  0.1441,  0.0247, -0.0079,
          -0.0102],
         [ 0.0728, -0.3231, -0.3207,  0.2351,  0.0593, -0.2558, -0.2599,
           0.1402],
         [ 0.0215,  0.0238, -0.0062,  0.0713,  0.0719, -0.3125,  0.2452,
           0.2295]],

        [[ 0.0297,  0.0304, -0.0537,  0.0507,  0.0519,  0.0536,  0.1714,
           0.1713],
         [ 0.0535, -0.2938, -0.2938,  0.2355,  0.0292, -0.0533, -0.0526,
           0.0499],
         [ 0.0535,  0.0534,  0.1716,  0.0528,  0.0517, -0.2940,  0.2354,
           0.2351]]], grad_fn=<ViewBackward0>)

: 

In [13]:
output = w_o(sim3)
# Projecting from key_dimension back to d_model (up projection)
print(output.shape)
output
# We have output tensor of the same dimension as our input. Ready to be passed on to the next layer.

NameError: name 'sim3' is not defined

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Compute Scaled Dot Product Attention using Multiple Heads
    """

    def __init__(self, d_model:int, num_heads: int):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.w_q = nn.Linear(self.d_model, self.d_model)
        self.w_k = nn.Linear(self.d_model, self.d_model)
        self.w_v = nn.Linear(self.d_model, self.d_model)
        self.w_o = nn.Linear(self.d_model, self.d_model)

        # Checking if d_model is divisible by num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.head_dim = d_model // num_heads # Dimension of query, key and value vectors within each head

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        
        # Checking if input has correct embedding dimension
        if input.shape[-1] != self.d_model:
            raise ValueError(f"Expected input feature dimension to be d_model ({self.d_model}), but got {input.shape[-1]}")

        # Input Dimension: (batch_size, num_tokens, d_model)
        batch_size = input.shape[0]
        num_tokens = input.shape[1]

        # Creating query, key and value vectors from input embeddings. 
        # These vectors contain queries from all the heads combined.
        q = self.w_q(input)
        k = self.w_k(input)
        v = self.w_v(input)
        # Dimension: (batch_sizee, num_tokens, d_model)

        # Splitting our vectors into heads by reshaping our tensors.
        q_head = q.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        k_head = k.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        v_head = v.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        # Dimension: (batch_size, num_tokens, num_heads, head_dimension)

        # Swapping (num_tokens) and (num_heads) dimensions.
        # We do this because we need (num_tokens) and (head_dimension) at the end.
        # This is because we need to perform matrix operations on the last two dimensions
        q_head = q_head.transpose(1, 2)
        k_head = k_head.transpose(1, 2)
        v_head = v_head.transpose(1, 2)
        # Dimension: (batch_size, num_heads, num_tokens, head_dimension)

        # Getting a transpose of key vectors for matrix multiplication 
        k_t = k_head.transpose(-2, -1)
        sim1 = (q_head @ k_t)/math.sqrt(self.head_dim) # Scaled dot product of query and key vectors
        # sim1 is the matrix of attention scores for each token
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Performing softmax along the keys dimension to get attention scores for each key
        sim2 = F.softmax(sim1, dim=-1)
        # Dimension: (batch_size, num_heads, num_queries, num_keys)

        # Weighted sum of attention weights and value vectors
        sim3 = sim2 @ v_head
        # Dimension: (batch_size, num_heads, num_queries, head_dimension)

        # Swapping (num_heads) and (num_queries) dimension back so that we can recombine all the heads
        sim3 = sim3.transpose(1, 2).contiguous()
        sim3 = sim3.view(batch_size, num_tokens, self.d_model)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        # Passing all of our heads into a final layer so that all heads can interact.
        output = self.w_o(sim3)
        # Dimension: (batch_size, num_tokens, d_model)
        # Same as input dimension
        
        return output

In [21]:
dummy_input = torch.rand(2, 3, 8)
dummy_input

tensor([[[0.3491, 0.8438, 0.3255, 0.2694, 0.0542, 0.5572, 0.2220, 0.1423],
         [0.7077, 0.3177, 0.8534, 0.0887, 0.6183, 0.9013, 0.0850, 0.0533],
         [0.0701, 0.6213, 0.4967, 0.5100, 0.4762, 0.7804, 0.5173, 0.5441]],

        [[0.5060, 0.7360, 0.2101, 0.1105, 0.7017, 0.6543, 0.7521, 0.3989],
         [0.3286, 0.7120, 0.0106, 0.5552, 0.4712, 0.1521, 0.8142, 0.8798],
         [0.9882, 0.7986, 0.6525, 0.7345, 0.9641, 0.8569, 0.0591, 0.9911]]])

In [22]:
mha_layer = MultiHeadAttention(d_model=8, num_heads=2)

In [23]:
output = mha_layer(dummy_input)